In [1]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
from pathlib import Path
import re
from io import BytesIO, TextIOWrapper
from tqdm import tqdm
from collections import defaultdict
from pathlib import Path
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.interpolate import griddata
from scipy import sparse
import math
import json
import time
import h5py
from pyproj import CRS, Transformer
import rasterio
from timezonefinder import TimezoneFinder
import pytz
from datetime import datetime, timedelta
import xarray as xr
import cdsapi
import zipfile
from dateutil.relativedelta import relativedelta
import dask
import dask.array
import xdem

In [2]:
print(xr.backends.list_engines())

{'scipy': <ScipyBackendEntrypoint>
  Open netCDF files (.nc, .cdf and .nc.gz) using scipy in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.ScipyBackendEntrypoint.html, 'cfgrib': <CfGribBackend>
  Open GRIB files (.grib, .grib2, .grb and .grb2) in Xarray
  Learn more at https://github.com/ecmwf/cfgrib, 'rasterio': <RasterioBackend>, 'store': <StoreBackendEntrypoint>
  Open AbstractDataStore instances in Xarray
  Learn more at https://docs.xarray.dev/en/stable/generated/xarray.backends.StoreBackendEntrypoint.html}


# Loading files

In [3]:
subset_feature_list = ['ID', 'lon', 'lat', 'fireday', 'year', 'DOB', 'firearea', 'cumuarea', 'prec', 'tmax', 'ws', 'rh', 
                           'dem', 'slope', 'aspect', 
                           'Biomass', 'Closure', 'prcB', 'prcC']

In [4]:
fire_growth_2024 = pd.read_csv('Fire growth points/Firegrowth_pts_v1_1_2024/Firegrowth_pts_v1_1_2024.csv', usecols=subset_feature_list)

In [5]:
def lonlat_to_canada_lambert(df, lon_col='lon', lat_col='lat'):

    """
    Adds the projected coordinates in meters
    
    :param df: Dataframe
    :param lon_col: The longitude column name
    :param lat_col: The latitude column name
    """

    if lon_col not in df.columns or lat_col not in df.columns:
        raise ValueError(f"DataFrame must contain columns '{lon_col}' and '{lat_col}'")

    source_crs = CRS.from_epsg(4269)   # NAD83 geographic (degrees)
    target_crs = CRS.from_epsg(3347)   # NAD83 / Canada Lambert (meters)

    transformer = Transformer.from_crs(source_crs, target_crs, always_xy=True)

    lons = df[lon_col].to_numpy(dtype=float)
    lats = df[lat_col].to_numpy(dtype=float)

    # Transform to meters coordinates
    eastings, northings = transformer.transform(lons, lats)

    df['easting'] = eastings
    df['northing'] = northings
    df.attrs['target_crs'] = target_crs.to_string()

    return df, target_crs

In [6]:
fire_growth_2024, used_crs = lonlat_to_canada_lambert(fire_growth_2024)

In [7]:
fire_growth_2024.head()

,ID,DOB,year,fireday,firearea,prec,tmax,ws,rh,Biomass,...,prcB,prcC,dem,slope,aspect,cumuarea,lon,lat,easting,northing
0,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,86.222221,...,13.000000,87.000000,303.444458,2.517298,34.278625,311.04,-116.106160,60.100154,4.920918e+06,2.890838e+06
1,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,89.888885,...,15.666667,84.333333,301.222229,2.387177,34.278625,311.04,-116.104586,60.100425,4.921008e+06,2.890834e+06
2,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,93.000000,...,46.777779,53.222221,302.000000,1.103502,74.148155,311.04,-116.110339,60.098556,4.920644e+06,2.890762e+06
3,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,99.666664,...,47.111111,52.888889,302.111115,1.740722,5.107979,311.04,-116.108765,60.098826,4.920734e+06,2.890757e+06
4,2024_1,225,2024,1,311.04,0.0018,32.062402,13.14911,35.655201,85.222221,...,24.000000,76.000000,306.666656,1.065100,5.107979,311.04,-116.107191,60.099097,4.920824e+06,2.890753e+06


In [8]:
len(np.unique(fire_growth_2024['ID'].values))

372

In [9]:
fire_growth_2024.shape

(6240195, 21)

# Fire Subsets

In [10]:
FIRE_SUBSETS_FOLDER = 'Fire_Subsets'
if not os.path.exists(FIRE_SUBSETS_FOLDER):
    os.makedirs(FIRE_SUBSETS_FOLDER)
    print(f"Created directory: {FIRE_SUBSETS_FOLDER}")

In [11]:
def save_fire_subsets(df, fire_ids, output_folder):
    
    """
    Save the target fires as separate csvs
    
    :param df: Dataframe
    :param fire_ids: List of fire IDs
    :param output_folder: Folder of fires' csvs
    """

    for fire_id in fire_ids:
        # Filter the dataframe for the specific ID
        subset = df[df['ID'] == fire_id]
        
        if not subset.empty:
            # Define the filename
            file_name = f"subset_fire_{fire_id}.csv"
            file_path = os.path.join(output_folder, file_name)
            
            # Save to CSV
            subset.to_csv(file_path, index=False)
            print(f"Saved: {file_path}")
        else:
            print(f"Warning: Fire ID {fire_id} not found in the dataframe.")

In [12]:
# Quick verification
subsets_ids = ['2024_188']
save_fire_subsets(fire_growth_2024, subsets_ids, FIRE_SUBSETS_FOLDER)

Saved: Fire_Subsets\subset_fire_2024_188.csv


# Topography

In [14]:
DEM_FOLDER = Path('DEM_API/Canada_DEM')
DEM_FOLDER.mkdir(parents=True, exist_ok=True)

## Preprocessing (with xdem)

In [15]:
def process_dem_folder_xdem(
    dem_original_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Elevation_original",
    dem_avg_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Elevation_average",
    dem_90m_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Elevation",
    slope_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Slope",
    aspect_dir=f"{DEM_FOLDER}/DEM_Files_XDEM/Aspect",
    scale_factor=3,
    target_resolution=90,
    resampling="average",
    slope_method="Horn",
):
    """
    Generate the DEM, slope, and aspect folders from the original DEM folders
    
    :param dem_original_dir: Original DEM folder
    :param dem_avg_dir: Target DEM folder (3x3 averaged blocs)
    :param dem_90m_dir: Target DEM folder (90m)
    :param slope_dir: Target slope folder (90m)
    :param aspect_dir: Target aspect folder (90m)
    :param scale_factor: Bloc size
    :param target_resolution: Final resolution
    :param resampling: Resampling method downsampling
    :param slope_method: Slope and aspect calculation algorithm
    """
    
    dem_original_dir = Path(dem_original_dir)
    dem_avg_dir = Path(dem_avg_dir)
    dem_90m_dir = Path(dem_90m_dir)
    slope_dir = Path(slope_dir)
    aspect_dir = Path(aspect_dir)

    # Create output folders
    dem_avg_dir.mkdir(parents=True, exist_ok=True)
    dem_90m_dir.mkdir(parents=True, exist_ok=True)
    slope_dir.mkdir(parents=True, exist_ok=True)
    aspect_dir.mkdir(parents=True, exist_ok=True)

    # Loop over all GeoTIFF DEMs
    for dem_path in dem_original_dir.glob("*.tif"):

        if dem_path.name != 'DEM_lat_51_61_lon_-132_-122.tif': # Added this so you do not waste your time processing all the files
            print(f"Skipping {dem_path.name}\n")
            continue
        
        print(f"Processing {dem_path.name}")

        # Load DEM
        dem = xdem.DEM(dem_path)

        # Average
        dem_avg = dem.reproject(
            res=dem.res[0] * scale_factor,
            resampling=resampling,
        )

        # Reproject to 90 m
        dem_90m_with_m = dem.reproject(
            crs="EPSG:3347", 
            res=target_resolution, 
            resampling=resampling
        )

        # Output filenames
        dem_avg_out = dem_avg_dir / f"{dem_path.stem}_avg{dem_path.suffix}"
        dem_90m_out = dem_90m_dir / f"{dem_path.stem}_90m{dem_path.suffix}"
        slope_out = slope_dir / f"{dem_path.stem.replace('DEM', 'SLOPE')}_90m{dem_path.suffix}"
        aspect_out = aspect_dir / f"{dem_path.stem.replace('DEM', 'ASPECT')}_90m{dem_path.suffix}"

        # Save DEM (average)
        # dem_avg.save(dem_90m_out, driver="GTiff")
        dem_avg.to_file(dem_avg_out, driver="GTiff")
        print('Average done')
        
        # Save DEM (90m)
        # dem_90m_with_m.save(dem_90m_out, driver="GTiff")
        dem_90m_with_m.to_file(dem_90m_out, driver="GTiff")
        print('Projection/Downscaling done')

        # Compute terrain attributes
        slope = xdem.terrain.slope(dem_90m_with_m, surface_fit=slope_method)
        aspect = xdem.terrain.aspect(dem_90m_with_m)

        # Save slope and aspect
        # slope.save(slope_out, driver="GTiff")
        # aspect.save(aspect_out, driver="GTiff")
        slope.to_file(slope_out, driver="GTiff")
        print('Slope done')
        aspect.to_file(aspect_out, driver="GTiff")
        print('Aspect done')

        print(f"DEM {dem_path} processed successfully.\n")

    print("All DEMs processed successfully.")

In [16]:
%%time
process_dem_folder_xdem()

Skipping DEM_lat_41_51_lon_-82_-72.tif

Processing DEM_lat_51_61_lon_-132_-122.tif
Average done
Projection/Downscaling done
Slope done
Aspect done
DEM DEM_API\Canada_DEM\DEM_Files_XDEM\Elevation_original\DEM_lat_51_61_lon_-132_-122.tif processed successfully.

Skipping DEM_lat_51_61_lon_-72_-62.tif

All DEMs processed successfully.
CPU times: total: 3min 46s
Wall time: 2min 58s


In [ ]:
# Quick verification
raster_path = "DEM_API/Canada_DEM/DEM_Files_XDEM/Elevation_original/DEM_lat_51_61_lon_-132_-122.tif"

with rasterio.open(raster_path) as src:
    bounds = src.bounds
    print(src.crs)
    print(f"Bounding box of {raster_path}:")
    print(f"Left (min X): {bounds.left}")
    print(f"Bottom (min Y): {bounds.bottom}")
    print(f"Right (max X): {bounds.right}")
    print(f"Top (max Y): {bounds.top}")

EPSG:4326
Bounding box of DEM_API/Canada DEM/DEM files XDEM/Elevation_original/DEM_lat_51_61_lon_-132_-122.tif:
Left (min X): -132.0001388888889
Bottom (min Y): 50.999861111111116
Right (max X): -121.99986111111112
Top (max Y): 61.00013888888889


## Pipeline (with xdem)

In [17]:
def get_overlapping_tiles(csv_min_lon, csv_max_lon, csv_min_lat, csv_max_lat, raster_folder):

    """
    Get all the DEM rasters necessary for a fire based on longitude and latitudes bounds for the fire
    
    :param csv_min_lon: Fire minimal longitude
    :param csv_max_lon: Fire maximal longitude
    :param csv_min_lat: Fire minimal latitude
    :param csv_max_lat: Fire maximal longitude
    :param raster_folder: Ratser folder
    """

    raster_paths = list(Path(raster_folder).glob("*.tif"))
    overlapping = []

    for rpath in raster_paths:
        # Example filename: DEM_lat_51_61_lon_-132_-122_90m.tif
        fname = rpath.stem 
        # Extract the min and max lon and lat
        m = re.search(r"lat_(-?\d+)_(-?\d+)_lon_(-?\d+)_(-?\d+)", fname)
        if not m:
            continue
        min_lat, max_lat, min_lon, max_lon = map(int, m.groups())

        # Check if CSV bounding box overlaps raster bounding box
        if (csv_max_lon >= min_lon and csv_min_lon <= max_lon and
            csv_max_lat >= min_lat and csv_min_lat <= max_lat):
            # overlapping.append(rpath.name)
            overlapping.append(rpath)

    return overlapping

In [19]:
# Quick verification
get_overlapping_tiles(csv_min_lon=-125, csv_max_lon=-126, csv_min_lat=58, csv_max_lat=59, raster_folder='DEM_API/Canada_DEM/DEM_Files_XDEM/Elevation')

[WindowsPath('DEM_API/Canada_DEM/DEM_Files_XDEM/Elevation/DEM_lat_51_61_lon_-132_-122_90m.tif')]

In [20]:
def sample_raster_for_points(raster_path, coords, output_array):

    """
    Get the values (DEM, slope, or aspect) from a specified raster corresponding 
    to the specidied coordinates
    
    :param raster_path: Raster path
    :param coords: List of coordinates
    :param output_array: values array
    """
    
    with rasterio.open(raster_path) as src:
        bounds = src.bounds
        # Mask points inside raster bounds
        mask = [(x >= bounds.left and x <= bounds.right and
                y >= bounds.bottom and y <= bounds.top) for x, y in coords]
        if any(mask):
            coords_in = [c for c, m in zip(coords, mask) if m]
            vals = np.array([v[0] for v in src.sample(coords_in)], dtype=float)
            # Handle nodata
            if src.nodata is not None:
                vals[vals == src.nodata] = np.nan
            # Assign sampled values back to correct positions
            j = 0
            for i, m in enumerate(mask):
                if m:
                    output_array[i] = vals[j]
                    j += 1

In [21]:
def add_dem_slope_aspect_bulk(df,
                              dem_avg_paths,
                              dem_paths,
                              slope_paths,
                              aspect_paths,
                              lat_col="lat",
                              lon_col="lon",
                              x_col="easting",
                              y_col="northing"):
    """
    Sample DEM, slope, and aspect from multiple raster files.
    For each point, it uses the raster tile that contains it.
    Points not inside any raster are set to NaN.
    
    :param df: dataframe 
    :param dem_paths: List of DEM raster (averaged) paths necessary for the fire
    :param dem_paths: List of DEM raster (90m) paths necessary for the fire
    :param slope_paths: List of slope raster paths necessary for the fire
    :param aspect_paths: List of aspect raster paths necessary for the fire
    :param lat_col: Name of the latitude column in the dataframe
    :param lon_col: Name of the longitude column in the dataframe
    :param x_col: Name of the meter x column in the dataframe
    :param y_col: Name of the meter y column in the dataframe
    """

    x_src = df[x_col]
    y_src = df[y_col]

    coords_lonlat = list(zip(df[lon_col].values, df[lat_col].values))
    coords_xy = list(zip(x_src, y_src))

    # Initialize output arrays
    dem_avg_vals = np.full(len(df), np.nan, dtype=float)
    dem_vals = np.full(len(df), np.nan, dtype=float)
    slope_vals = np.full(len(df), np.nan, dtype=float)
    aspect_vals = np.full(len(df), np.nan, dtype=float)

    # Sample DEM tiles (lat/lon)
    for dem_avg_path in dem_avg_paths:
        sample_raster_for_points(dem_avg_path, coords_lonlat, dem_avg_vals)
    # Sample 90m DEM tiles (lat/lon)
    for dem_path in dem_paths:
        sample_raster_for_points(dem_path, coords_xy, dem_vals)
    # Sample Slope tiles (m)
    for slope_path in slope_paths:
        sample_raster_for_points(slope_path, coords_xy, slope_vals)
    # Sample Aspect tiles (m)
    for aspect_path in aspect_paths:
        # sample_raster_for_points(aspect_path, coords_lonlat, aspect_vals)
        sample_raster_for_points(aspect_path, coords_xy, aspect_vals)

    # Assign to DataFrame
    df["dem_v2"] = dem_avg_vals
    df["dem_v3"] = dem_vals
    df["slope_v2"] = slope_vals
    df["aspect_v2"] = aspect_vals

    return df

In [23]:
FIRE_SUBSETS_FOLDER = 'Fire_Subsets' 
DEM_FOLDER = "DEM_API/Canada_DEM"
ELEVATION_FOLDER = f"{DEM_FOLDER}/DEM_Files_XDEM/Elevation"
ELEVATION_AVG_FOLDER = f"{DEM_FOLDER}/DEM_Files_XDEM/Elevation_average"
SLOPE_FOLDER = f"{DEM_FOLDER}/DEM_Files_XDEM/Slope"
ASPECT_FOLDER = f"{DEM_FOLDER}/DEM_Files_XDEM/Aspect"

In [25]:
%%time

for csv_file in Path(FIRE_SUBSETS_FOLDER).glob("*.csv"):

    if csv_file.name != 'subset_fire_2024_188.csv': # Keeping only the target fire (to remove later)
        print(f"Skipping {csv_file.name}\n")
        continue

    print(f"Processing {csv_file.name}:")
    
    df = pd.read_csv(csv_file)

    # Get bounding box
    min_lon, max_lon = df["lon"].min(), df["lon"].max()
    min_lat, max_lat = df["lat"].min(), df["lat"].max()

    print(min_lon, max_lon)
    print(min_lat, max_lat)

    # Get overlapping DEM tiles
    dem_avg_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, ELEVATION_AVG_FOLDER)
    dem_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, ELEVATION_FOLDER)
    slope_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, SLOPE_FOLDER)
    aspect_tiles = get_overlapping_tiles(min_lon, max_lon, min_lat, max_lat, ASPECT_FOLDER)
    print(dem_avg_tiles)
    print(dem_tiles)
    print(slope_tiles)
    print(aspect_tiles)

    if not dem_tiles:
        print(f"No DEM tile found for {csv_file.name}, skipping...")
        continue

    df_aug = add_dem_slope_aspect_bulk(
        df,
        dem_avg_paths=dem_avg_tiles,
        dem_paths=dem_tiles,
        slope_paths=slope_tiles,
        aspect_paths=aspect_tiles,
        lat_col="lat",
        lon_col="lon",
        x_col="easting",
        y_col="northing"
    )

    # Save CSV
    df_aug.to_csv(Path(FIRE_SUBSETS_FOLDER)/csv_file.name, index=False)

    del df, df_aug

    print('\n')

Processing subset_fire_2024_188.csv:
-125.651308330267 -125.435873318329
58.1485577612542 58.2140934603227
[WindowsPath('DEM_API/Canada_DEM/DEM_Files_XDEM/Elevation_average/DEM_lat_51_61_lon_-132_-122_avg.tif')]
[WindowsPath('DEM_API/Canada_DEM/DEM_Files_XDEM/Elevation/DEM_lat_51_61_lon_-132_-122_90m.tif')]
[WindowsPath('DEM_API/Canada_DEM/DEM_Files_XDEM/Slope/SLOPE_lat_51_61_lon_-132_-122_90m.tif')]
[WindowsPath('DEM_API/Canada_DEM/DEM_Files_XDEM/Aspect/ASPECT_lat_51_61_lon_-132_-122_90m.tif')]


Skipping subset_fire_2024_560.csv

CPU times: total: 2.02 s
Wall time: 2.09 s


In [26]:
for filename in os.listdir(FIRE_SUBSETS_FOLDER):
    
    # Construct the full path
    file_path = os.path.join(FIRE_SUBSETS_FOLDER, filename)
    
    if os.path.isfile(file_path):
        
        print(f"Processing: {filename}")
        subset_fire_df = pd.read_csv(file_path)
        print(subset_fire_df.shape)

Processing: subset_fire_2024_188.csv
(4863, 25)
Processing: subset_fire_2024_560.csv
(35375, 32)


In [27]:
# Quick verification
subset_fire_df = pd.read_csv(f'{FIRE_SUBSETS_FOLDER}/subset_fire_2024_188.csv')
# subset_fire_df[subset_fire_df['fireday']==6][['DOB', 'dem', 'dem_v2', 'slope', 'slope_v2', 'aspect', 'aspect_v2']]
pd.options.display.float_format = '{:.2f}'.format
subset_fire_df[subset_fire_df['fireday']==6][['ID', 'fireday', 'lon', 'lat', 'easting', 'northing', 'DOB', 'dem', 'dem_v2', 'dem_v3']].head(20)

,ID,fireday,lon,lat,easting,northing,DOB,dem,dem_v2,dem_v3
19,2024_188,6,-125.61,58.21,4356393.25,2940386.59,203,1737.11,1736.00,1727.00
35,2024_188,6,-125.61,58.21,4356298.93,2940301.13,203,1644.00,1682.00,1640.00
36,2024_188,6,-125.61,58.21,4356388.82,2940296.70,203,1690.89,1682.00,1679.00
37,2024_188,6,-125.61,58.21,4356478.71,2940292.27,203,1687.00,1688.00,1700.00
56,2024_188,6,-125.61,58.21,4356204.61,2940215.67,203,1576.22,1570.00,1576.00
57,2024_188,6,-125.61,58.21,4356294.50,2940211.24,203,1602.00,1620.00,1598.00
58,2024_188,6,-125.61,58.21,4356384.39,2940206.81,203,1640.56,1633.00,1640.00
59,2024_188,6,-125.61,58.21,4356474.28,2940202.38,203,1669.67,1672.00,1676.00
82,2024_188,6,-125.61,58.21,4356200.18,2940125.78,203,1539.22,1530.00,1536.00
83,2024_188,6,-125.61,58.21,4356290.07,2940121.35,203,1553.67,1566.00,1549.00


In [28]:
def calculate_relative_error(df, column_pairs):
    error_df = pd.DataFrame(index=df.index)
    
    for actual_col, pred_col in column_pairs:
        error_name = f"rel_error_{pred_col}"
        
        valid_mask = df[actual_col].notna() & df[pred_col].notna()
        
        # Initialize the column with NaN
        error_df[error_name] = np.nan
        
        error_df.loc[valid_mask, error_name] = np.where(
            (df.loc[valid_mask, actual_col] == 0) & (df.loc[valid_mask, pred_col] == 0), 
            0.0,
            np.where(
                (df.loc[valid_mask, actual_col] == 0) & (df.loc[valid_mask, pred_col] > 0), 
                np.nan,
                ((df.loc[valid_mask, pred_col] - df.loc[valid_mask, actual_col]).abs() / df.loc[valid_mask, actual_col].abs()) * 100
            )
        )
        
    return error_df

In [29]:
# pairs = [('dem', 'dem_v2'), ('dem', 'dem_v3'), ('slope', 'slope_v2'), ('aspect', 'aspect_v2')]
pairs = [('dem', 'dem_v2'), ('dem', 'dem_v3')]
results = calculate_relative_error(subset_fire_df, pairs)
summary_stats = results.describe(percentiles=[0.5])
summary_stats_rounded = summary_stats.round(6)
summary_stats_rounded

,rel_error_dem_v2,rel_error_dem_v3
count,4863.00,4863.00
mean,0.61,0.69
std,0.55,0.62
min,0.00,0.00
50%,0.45,0.50
max,4.05,4.66


In [30]:
def calculate_slope_summary(y_true, y_pred):
    """
    Calculates Slope metrics and returns a summary DataFrame
    including MAE, RMSE, and error extremes.
    """
    # Convert to numpy arrays to ensure vectorized math works
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Remove any NaN pairs
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    t = y_true[mask]
    p = y_pred[mask]
    
    if len(t) == 0:
        return pd.DataFrame([{"Error": "No valid data points"}])

    # Calculate Linear Errors
    errors = np.abs(t - p)
    
    # Statistical Metrics
    mae = np.mean(errors)
    rmse = np.sqrt(np.mean(errors**2))
    max_err = np.max(errors)
    std_err = np.std(t - p)
    median_err = np.median(errors)

    summary_df = pd.DataFrame({
        "Metric": ["MAE", "RMSE", "Median Error", "Std Dev", "Sample Count"],
        "Value": [mae, rmse, median_err, std_err, len(t)]
    })

    return summary_df.round(4)

In [31]:
calculate_slope_summary(subset_fire_df['dem'], subset_fire_df['dem_v2'])

,Metric,Value
0,MAE,7.26
1,RMSE,10.02
2,Median Error,5.22
3,Std Dev,10.02
4,Sample Count,4863.00


In [32]:
calculate_slope_summary(subset_fire_df['slope'], subset_fire_df['slope_v2'])

,Metric,Value
0,MAE,2.32
1,RMSE,3.08
2,Median Error,1.81
3,Std Dev,2.97
4,Sample Count,4863.00


In [33]:
def calculate_aspect_summary(y_true, y_pred):
    """
    Calculates circular Aspect metrics and returns a summary DataFrame
    including MAE, RMSE, and error extremes.
    """
    # Convert to arrays and drop NaNs
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    t = y_true[mask]
    p = y_pred[mask]
    
    if len(t) == 0:
        return pd.DataFrame([{"Error": "No valid data points"}])

    # Circular Difference Logic (Shortest distance around 360)
    diff = np.abs(t - p) % 360
    circular_errors = np.where(diff > 180, 360 - diff, diff)

    # Calculate Statistics
    mae = np.mean(circular_errors)
    rmse = np.sqrt(np.mean(circular_errors**2))
    max_err = np.max(circular_errors)
    std_err = np.std(circular_errors)
    median_err = np.median(circular_errors)

    summary_df = pd.DataFrame({
        "Metric": ["MAE", "RMSE", "Median Error", "Std Dev", "Sample Count"],
        "Value": [mae, rmse, median_err, std_err, len(t)]
    })

    return summary_df.round(4)

In [34]:
calculate_aspect_summary(subset_fire_df['aspect'], subset_fire_df['aspect_v2'])

,Metric,Value
0,MAE,18.85
1,RMSE,31.11
2,Median Error,11.21
3,Std Dev,24.75
4,Sample Count,4863.00
